## **Understanding how state and context interact during execution**

You can access runtime information in tools, as well as via custom agent middleware.


```
** Input: "Schedule a meeting with John tomorrow"

** Tool 1: Extract Info
runtime.state["event"] = {
    "name": "Meeting with John",
    "date": "tomorrow"
}

** Tool 2: Create Meeting
event = runtime.state["event"]
user_id = runtime.context["user_id"]
create_meeting_api(user_id, event)
```

## **Static Runtime Context**

1. Specify the **context_schema**: This defines the structure of context stored in the agent runtime
2. Provide the context_schema to the agent during **create_agent()**
3. In the tool call, use the **runtime** parameter (typed as **ToolRuntime**).
4. Finally, when you invoke the agent, you can provide your dependencies like db connection to the agent in the **context** argument.

In [ ]:
from langchain_openai import ChatOpenAI

# Setup API Key
f = open('keys/.openai_api_key.txt')
OPENAI_API_KEY = f.read()

openai_chat_model = ChatOpenAI(api_key=OPENAI_API_KEY, 
                               model="gpt-4o-mini", 
                               temperature=1)

In [ ]:
# from dataclasses import dataclass
from langchain.agents import create_agent

# Step 1: Define the ContextSchema
class ContextSchema:
    favourite_color: str = "blue"
    language: str = "english"

# Step 2: Pass the ContextSchema to the agent
agent = create_agent(
    model=openai_chat_model,
    context_schema=ContextSchema
)

# Step 3: Pass the object during the agent invokation
response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is my favourite color?"}]},
    context=ContextSchema()
)

print(response.keys())

In [ ]:
for msg in response["messages"]:
    msg.pretty_print()

### **Access the Static Runtime Context**

Runtime context is not directly passed to the LLM, instead it is passed to the ToolRuntime. 

In [ ]:
# from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime

# Step 1: Define the ContextSchema
# @dataclass
class ContextSchema:
    favourite_color: str = "blue"
    language: str = "english"

@tool
def get_favourite_color(runtime: ToolRuntime) -> str:
    """Get user's favourite color"""
    return runtime.context.favourite_color
    

@tool
def get_language(runtime: ToolRuntime) -> str:
    """Get language of the user"""
    return runtime.context.language

# Step 2: Pass the ContextSchema to the agent
agent = create_agent(
    model=openai_chat_model,
    tools=[get_favourite_color, get_language],
    context_schema=ContextSchema
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is my favourite color?"}]},
    context=ContextSchema()  
)

print(response.keys())

In [ ]:
for msg in response["messages"]:
    msg.pretty_print()

In [ ]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is my favourite color? also tell me my preferred language?"}]},
    context=ContextSchema()  
)

for msg in response["messages"]:
    msg.pretty_print()